# Tutorial 05: Multi-Agent Workflows (Basics)

## Learning Objectives
After completing this notebook, you will be able to:
- Use Agent Framework's **built-in workflow builders** for multi-agent coordination
- Build **sequential workflows** with `SequentialBuilder` (agent chains)
- Build **concurrent workflows** with `ConcurrentBuilder` (parallel execution)
- Understand when to use each basic workflow pattern
- Learn the foundations for advanced patterns (covered in Tutorial 07)

## Key Concepts

### Why Multi-Agent Systems?

**Single Agent (Tutorials 1-5):**
- Simple, one agent handles everything
- Can become overloaded with too many responsibilities
- No specialization or parallelization
- Example: General-purpose assistant

**Multi-Agent Workflows:**
- Multiple specialized agents collaborate
- Each agent focuses on their area of expertise
- Parallel execution for speed
- Clear separation of concerns
- Example: Flight expert + Hotel expert + Activities expert

### Agent Framework Workflow Patterns

Agent Framework provides **built-in workflow builders**:

**Basic Patterns (this tutorial):**

1. **`SequentialBuilder`** - Agents work in order (A → B → C)
   - Shared conversation context passes through each agent
   - Each agent builds on the previous agent's output
   - Use for: Review/refinement, multi-step tasks

2. **`ConcurrentBuilder`** - Agents work in parallel (A, B, C simultaneously)
   - Fan-out with the same input to all agents
   - Fan-in aggregating results from all agents
   - Use for: Independent tasks, gathering multiple perspectives

**Advanced Patterns (Tutorial 07):**

3. **`WorkflowBuilder`** - Custom DAG with executors and edges
   - Full control over the workflow graph
   - Custom routing logic and conditions
   - Use for: Complex business logic, conditional flows

4. **`MagenticBuilder`** - AI-powered orchestration
   - LLM creates and manages execution plans
   - Dynamic agent selection and coordination
   - Use for: Complex, unpredictable tasks

### Benefits of Framework Workflows

**Built-in coordination** - No manual orchestration code needed  
**Automatic aggregation** - Results are automatically combined  
**Event streaming** - Track progress in real-time  
**Checkpointing** - Pause/resume long workflows  
**Visualization** - Display workflow graphs  
**Production-ready** - Built-in error handling and observability

---

## Step 1: Setup and Imports

In [ ]:
import asyncio
from typing import cast

from agent_framework import (
    # Workflow events for progress tracking
    WorkflowEvent,
    WorkflowEventType,
    # Basic types
    Message,
    WorkflowViz,
)
from agent_framework.orchestrations import (
    # Basic workflow builders
    SequentialBuilder,
    ConcurrentBuilder,
)
from agent_framework.azure import AzureOpenAIChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)
print("✅ Import successful!")
print("📦 Workflow builders ready: SequentialBuilder, ConcurrentBuilder")
print("💡 For advanced patterns (WorkflowBuilder, MagenticBuilder), see Tutorial 07!")

In [ ]:
import os
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication method selection ===
# True: API key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("🔑 Authentication method: API key authentication")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework automatically reads AZURE_OPENAI_API_KEY from environment variables
    # and prioritizes it over credential, so explicitly remove it
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Authentication method: DefaultAzureCredential")

print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Setting Up Tracer with OpenTelemetry
OpenTelemetry tracers are useful for debugging multi-agent systems.
In this environment, the Agent Framework does not provide `setup_observability`,
so we manually set up the OpenTelemetry `TracerProvider` (including Exporter/Processor) and enable instrumentation with `enable_instrumentation()`.
Here we use OTLP gRPC (e.g., Jaeger / OpenTelemetry Collector on port `4317`) as the trace destination.

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os

from agent_framework.observability import configure_otel_providers, get_tracer

service_name = "agent_framework"
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

# Configure via environment variables (Agent Framework reads standard OTEL_* variables)
os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")
os.environ.setdefault("ENABLE_INSTRUMENTATION", "true")
os.environ.setdefault("OTEL_METRICS_EXPORTER", "none")  # Disable Metrics (change as needed)

# Specify enable_sensitive_data=True to enable collection of sensitive data (OpenAI IN/OUT data)
configure_otel_providers(enable_sensitive_data=True)

tracer = get_tracer()
print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")

## Step 2: Create Specialized Agents

Let's create domain expert agents to coordinate using workflows.

In [ ]:
async def create_travel_agents():
    """
    Create specialized travel planning agents.
    """
    # Create Azure OpenAI client
    chat_client = AzureOpenAIChatClient(
        **auth_kwargs,
        endpoint=azure_openai_endpoint,
        api_version=api_version,
        deployment_name=azure_deployment,
    )

    # Flight expert
    flight_agent = chat_client.as_agent(
        instructions="""
        You are a specialized flight booking specialist.
        Provide concise and practical flight recommendations considering:
        - Best time to book
        - Airline preferences and quality
        - Connection strategies
        - Price vs. convenience trade-offs
        
        Keep your response concise (max 2-3 sentences).
        """,
        name="FlightExpert",
    )
    
    # Hotel expert
    hotel_agent = chat_client.as_agent(
        instructions="""
        You are a specialized hotel booking specialist.
        Provide concise hotel recommendations considering:
        - Best areas for tourists
        - Cost-performance ratio
        - Proximity to attractions and transportation
        - Hotel quality and amenities
        
        Keep your response concise (max 2-3 sentences).
        """,
        name="HotelExpert",
    )
    
    # Activities expert
    activities_agent = chat_client.as_agent(
        instructions="""
        You are a local activities and experiences expert.
        Provide concise activity recommendations considering:
        - Must-see attractions
        - Popular local spots and hidden gems
        - Cultural experiences
        - Dining and gourmet options
        
        Keep your response concise (max 2-3 sentences).
        """,
        name="ActivitiesExpert",
    )
    
    return flight_agent, hotel_agent, activities_agent

print("✅ Agent factory created")

In [ ]:
from __future__ import annotations
import sys
from contextlib import nullcontext
from typing import Optional, cast
from agent_framework import AgentResponse, AgentResponseUpdate, Message, WorkflowEvent

async def run_stream_pretty(
    workflow,
    task: str,
    *,
    tracer=None,
    span_name: str = "SequentialBuilder",
    print_final: bool = True,
) -> list[Message]:
    """
    
    Executes workflow.run(task, stream=True) and displays streaming output
    in real-time within a Jupyter Notebook.

    Prerequisites (when using GroupChatBuilder):
      Build with GroupChatBuilder(..., intermediate_outputs=True).
      With the default (False), only the orchestrator's output is yielded,
      and participant AgentResponseUpdates are filtered out.

    Display:
      - AgentResponseUpdate → Tokens displayed incrementally (sys.stdout.write + flush)
      - Executor switch → Newline + name header
      - Final list[Message] → Displayed as CONVERSATION LOG summary

    Note:
      ★ Uses start_as_current_span() to correctly nest workflow internal spans
      as children of this span.
      _process_events is a regular async function, not an async generator,
      so there are no GeneratorExit issues when combined with context managers.
    """
    final_conversation: list[Message] = []
    last_executor_id: Optional[str] = None

    def _write(text: str) -> None:
        """Write with guaranteed flush for Jupyter"""
        sys.stdout.write(text)
        sys.stdout.flush()

    async def _process_events(workflow, task):
        nonlocal final_conversation, last_executor_id
        async for event in workflow.run(task, stream=True):
            if not isinstance(event, WorkflowEvent):
                continue

            # ----- Executor switch notification -----
            if event.type == "executor_invoked":
                eid = event.executor_id
                if eid != last_executor_id:
                    if last_executor_id is not None:
                        _write("\n")
                    _write(f"🤖 {eid}: ")
                    last_executor_id = eid

            # ----- Output event -----
            elif event.type == "output":
                data = event.data

                # (1) Streaming chunk: AgentResponseUpdate
                if isinstance(data, AgentResponseUpdate):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    chunk = data.text or ""
                    if chunk:
                        _write(chunk)

                # (2) Non-streaming response: AgentResponse
                elif isinstance(data, AgentResponse):
                    eid = event.executor_id
                    if eid != last_executor_id:
                        if last_executor_id is not None:
                            _write("\n")
                        _write(f"🤖 {eid}: ")
                        last_executor_id = eid
                    text = data.text or ""
                    if text:
                        _write(text)

                # (3) Final conversation log: list[Message]
                elif isinstance(data, list):
                    final_conversation = cast(list[Message], data)

    # ★ Use start_as_current_span() to nest workflow internal spans as children.
    #   _process_events is a regular async function, so no context conflicts occur.
    cm = tracer.start_as_current_span(span_name) if tracer else nullcontext()
    with cm:
        await _process_events(workflow, task)

    # Newline at end of stream
    _write("\n")

    if print_final and final_conversation:
        print("\n" + "=" * 80)
        print("CONVERSATION LOG")
        print("=" * 80)
        for msg in final_conversation:
            author = getattr(msg, "author_name", None) or msg.role
            text = msg.text or str(msg)
            print(f"\n[{author}]")
            print(text)
            print("-" * 80)

    return final_conversation

In [ ]:
from agent_framework import WorkflowViz
from IPython.display import SVG, display
import os

def visualize_workflow(workflow, filename="workflow_diagram", show_preview=True):
    """
    Output workflow graph information, save as SVG, and show preview
    """
    viz = WorkflowViz(workflow)
    
    try:
        svg_path = viz.export(format="svg", filename=filename)
        print("=" * 60)
        print(f"✅ SVG file saved: {svg_path}")
        print("=" * 60)
        print()
        
        if show_preview and os.path.exists(svg_path):
            display(SVG(filename=svg_path))
        
        return svg_path
        
    except ImportError as e:
        print("❌ Error: 'graphviz' package is not installed")
        print("Install with: pip install agent-framework[viz] --pre")
        print(f"Details: {e}")
        return None
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return None

## Step 3: Sequential Workflow - SequentialBuilder

The **sequential pattern** chains agents so each builds on the previous agent's output.

**Use cases:**
- Writing → Review → Editing
- Research → Analysis → Recommendations
- Draft → Critique → Refinement

The shared conversation passes through each participant sequentially.

In [ ]:
async def sequential_workflow_demo():
    """
    Demonstrates SequentialBuilder: agents work in order.
    Each agent sees the full conversation history from previous agents.
    """
    print("=== Sequential Workflow Pattern ===")
    print("Agents work in order: Flight → Hotel → Activities")
    print("Each agent builds on the previous agent's recommendations\n")
    
    flight_agent, hotel_agent, activities_agent = await create_travel_agents()
    
    # Build sequential workflow using SequentialBuilder
    workflow = SequentialBuilder(
        participants=[flight_agent, hotel_agent, activities_agent]
    ).build()
    
    print("🚀 Running sequential workflow...\n")
    
    # Run workflow and collect outputs
    outputs: list[list[Message]] = []
    async for event in workflow.run(
        "Plan a weekend trip to Paris. I'd like flights, hotels, and cultural activities.",
        stream=True,
    ):
        # Track agent progress
        if event.type == "executor_invoked":
            print(f"   🤖 {event.executor_id} is working...")
        
        # Collect final outputs
        if event.type == "output":
            outputs.append(cast(list[Message], event.data))
    
    # Display final conversation
    if outputs:
        print(f"\n{'='*60}")
        print("📋 Final Conversation (Sequential Flow)")
        print(f"{'='*60}\n")
        
        for i, msg in enumerate(outputs[-1], start=1):
            name = msg.author_name or msg.role
            print(f"{'-'*60}")
            print(f"{i}. [{name}]")
            print(f"{'-'*60}")
            print(f"{msg.text}\n")
        
        print(f"{'='*60}")
        print("✅ Sequential workflow complete!")
        print(f"{'='*60}")
        print("Benefits:")
        print("✓ Each agent builds on previous recommendations")
        print("✓ Shared conversation context")
        print("✓ No manual coordination code needed")
        print("✓ Clear sequential flow")

await sequential_workflow_demo()

In [ ]:

flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# Build sequential workflow using SequentialBuilder
# ★ intermediate_outputs=True: Yields each agent's AgentResponseUpdate (token chunks)
#   as output events. With False (default), only the "end" executor's
#   list[Message] passes through, and no streaming output is displayed.
workflow = SequentialBuilder(
    participants=[flight_agent, hotel_agent, activities_agent],
    intermediate_outputs=True,
).build()

# Visualize workflow using the helper function
svg_path = visualize_workflow(
    workflow, 
    filename="sequential_multi_agent_workflow.svg",
    show_preview=True
)

In [ ]:
task = "Plan a weekend trip to Paris. I'd like flights, hotels, and cultural activities."

# Run with the stream helper (stream display + collect final conversation)
final_conversation = await run_stream_pretty(workflow, task, tracer=tracer)

In [ ]:
for msg in final_conversation:
    print(msg.to_dict())

## Step 4: Concurrent Workflow - ConcurrentBuilder

The **concurrent pattern** runs agents in parallel with automatic fan-out/fan-in.

**Use cases:**
- Gather multiple perspectives simultaneously
- Independent parallel tasks
- Researcher + Marketer + Legal review (different domains)

All agents receive the same input and work simultaneously.

In [ ]:

"""
Demonstrates ConcurrentBuilder: agents work in parallel.
All agents receive the same prompt simultaneously.
Results are automatically aggregated.
"""
print("=== Concurrent Workflow Pattern ===")
print("Agents work in parallel: Flight ∥ Hotel ∥ Activities")
print("All agents receive the same input, work simultaneously\n")

flight_agent, hotel_agent, activities_agent = await create_travel_agents()

# Build concurrent workflow using ConcurrentBuilder
# ★ intermediate_outputs=True: Yields each agent's AgentResponseUpdate (token chunks)
#   as output events. With False (default), only the "aggregator"'s
#   list[Message] passes through, and no streaming output is displayed.
workflow = ConcurrentBuilder(
    participants=[flight_agent, hotel_agent, activities_agent],
    intermediate_outputs=True,
).build()


# Visualize workflow using the helper function
svg_path = visualize_workflow(
    workflow, 
    filename="concurrent_multi_agent_workflow.svg",
    show_preview=True
)

In [ ]:

print("🚀 Running concurrent workflow (all agents in parallel)...\n")

# ★ Use start_as_current_span() to nest workflow internal spans as children.
with tracer.start_as_current_span("ConcurrentBuilder"):
    # Run workflow and collect outputs
    outputs: list[list[Message]] = []
    async for event in workflow.run(
        "I'm planning a 3-day trip to Tokyo. Please provide recommendations for flights, hotels, and things to do.",
        stream=True,
    ):
        # Track agent progress
        if event.type == "executor_invoked":
            print(f"   🤖 {event.executor_id} is working...")
        
        # Collect final outputs
        if event.type == "output":
            outputs.append(cast(list[Message], event.data))

    # Display final conversation
    if outputs:
        print(f"\n{'='*60}")
        print("📋 Final Conversation (Concurrent Flow)")
        print(f"{'='*60}\n")
        
        for i, msg in enumerate(outputs[-1], start=1):
            name = msg.author_name or msg.role
            print(f"{'-'*60}")
            print(f"{i}. [{name}]")
            print(f"{'-'*60}")
            print(f"{msg.text}\n")

## Workflow Pattern Comparison

### When to Use Each Pattern

| Pattern | Builder | Use Case | Pros | Cons |
|---------|---------|----------|------|------|
| **Sequential** | `SequentialBuilder` | Review/refinement, multi-step tasks | Simple, clear flow | Slower, no parallelism |
| **Concurrent** | `ConcurrentBuilder` | Independent tasks, multiple perspectives | Fast, parallel | No sequential dependencies |

### Real-World Examples

**Sequential:**
- Documents: Draft → Review → Edit → Approve
- Research: Gather → Analyze → Summarize → Recommend
- Code: Create → Test → Review → Deploy

**Concurrent:**
- Product launch: Marketing + Legal + Engineering (parallel review)
- Research: Multiple researchers investigating different aspects
- Analysis: Technical + Business + Legal perspectives

### Advanced Patterns (Tutorial 07)

For more complex scenarios, see **Tutorial 07: Advanced Workflows**:
- **WorkflowBuilder** - Custom DAG, conditional routing, validation gates
- **MagenticBuilder** - AI-powered orchestration, dynamic planning

## Key Takeaways

### What You Learned

1. **Use Built-in Workflow Builders**
   - Don't write manual coordination code!
   - `SequentialBuilder` for sequential tasks
   - `ConcurrentBuilder` for parallel execution

2. **Sequential Workflows**
   - Agents work in order: A → B → C
   - Each agent sees the full conversation history
   - Best for refinement, multi-step processes
   - Simple `SequentialBuilder(participants=[agents]).build()` API

3. **Concurrent Workflows**
   - Agents work in parallel: A ∥ B ∥ C
   - Automatic fan-out and aggregation
   - Best for independent tasks, speed
   - Results are automatically combined

4. **Workflows Handle Complexity**
   - Automatic coordination
   - Built-in event streaming
   - Error handling and observability
   - Production-ready patterns

### Production Patterns

```python
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder

# Sequential: Review workflow
workflow = SequentialBuilder(
    participants=[writer, editor, approver]
).build()

# Concurrent: Parallel analysis
workflow = ConcurrentBuilder(
    participants=[technical_analyst, business_analyst, legal_analyst]
).build()

# Run workflow (stream=True to stream WorkflowEvents)
async for event in workflow.run("your prompt", stream=True):
    if event.type == "output":
        print(event.data)
```

## Exercises

1. **Content Creation Pipeline**
   - Sequential: Idea Generator → Writer → Editor → SEO Optimizer
   - Each agent refines the content

2. **Product Analysis**
   - Concurrent: Technical Reviewer ∥ Market Analyst ∥ Competitive Researcher
   - Get multiple perspectives simultaneously

3. **Code Review**
   - Sequential: Linter → Security Checker → Performance Analyzer → Approver
   - Each step builds on the previous checks

4. **Customer Feedback Analysis**
   - Concurrent: Sentiment Analyst ∥ Feature Extraction ∥ Priority Scorer
   - Analyze different aspects in parallel

In [ ]:
# Exercise: Create a content creation workflow

async def content_creation_exercise():
    """
    Create a sequential workflow for content creation:
    Idea Generator → Writer → Editor → SEO Optimizer
    """
    # Implement here!
    # 1. Create 4 specialized agents
    # 2. Build sequential workflow with SequentialBuilder(participants=[agents]).build()
    # 3. Run with a topic: workflow.run("topic", stream=True)
    # 4. Display the final refined content
    pass

async def product_analysis_exercise():
    """
    Create a concurrent workflow for product analysis:
    Technical ∥ Market ∥ Competitive (parallel)
    """
    # Implement here!
    # 1. Create 3 specialized analyst agents
    # 2. Build concurrent workflow with ConcurrentBuilder(participants=[agents]).build()
    # 3. Run with a product description: workflow.run("product", stream=True)
    # 4. Display the aggregated analysis
    pass

print("Exercises ready - implement sequential and concurrent workflows!")

## Next Steps

Congratulations! You've mastered the basic multi-agent workflow patterns!

You can now:
- Build sequential agent workflows (chains)
- Create concurrent parallel execution
- Use built-in workflow builders
- Track workflow progress with events

---

### Quick Reference

**Sequential Workflow:**
```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(
    participants=[agent1, agent2, agent3]
).build()

# Streaming execution
async for event in workflow.run("your prompt", stream=True):
    if event.type == "output":
        print(event.data)
```

**Concurrent Workflow:**
```python
from agent_framework.orchestrations import ConcurrentBuilder

workflow = ConcurrentBuilder(
    participants=[agent1, agent2, agent3]
).build()

# Run and get outputs
result = await workflow.run("your prompt")
outputs = result.get_outputs()
```

**Pattern Selection:**
- **Sequential** → When order matters, agents build on each other
- **Concurrent** → When speed matters, independent tasks
- **Custom** → See Tutorial 07 for WorkflowBuilder
- **Magentic** → See Tutorial 07 for AI orchestration